# NMax3VW — Normalized Maximum 3rd Ventricle Width (MRI Adaptation)

**NMax3VW** is the Normalized Maximum 3rd Ventricle Width: the maximum width of the 3rd ventricle
measured on the axial plane at the AC–PC level, divided by the total brain width on the same slice.

## Anatomical context

The **3rd ventricle** is a narrow, slit-like midline cavity located between the two thalami.
It communicates anteriorly with the lateral ventricles via the **Foramen of Monro** and posteriorly
with the 4th ventricle via the cerebral aqueduct. In healthy adults the 3rd ventricle is very narrow
(≈2–4 mm). Dilation indicates obstructive or communicating hydrocephalus, and is commonly seen in
TBI, normal-pressure hydrocephalus, and neurodegenerative disease.

## Measurement definition

1. Identify the **AC–PC axial slice** — the plane containing both the anterior commissure (AC) and
   posterior commissure (PC). This is the standard reference plane for subcortical anatomy.
2. Scan ±10 slices superiorly from that plane.
3. On each candidate slice, isolate the **3rd ventricle ROI**: the midline strip between the
   AC column (aj) and PC column (pj) in the image coordinate frame.
4. Measure the width of the 3rd ventricle on every row of the ROI (number of bright CSF pixels
   per row after thresholding), take the **median** as the per-slice width.
5. **Normalize** by total brain width on the same slice.
6. Report the **maximum** normalized width across all candidate slices → **NMax3VW**.

## MRI adaptation notes

This notebook adapts [Sharada Samadani's CT-based NMax3VW pipeline](https://github.com/samadanilab/Ventriculomegaly-Feature-Computation)
for T1-weighted MRI.

Key differences from the CT version:
- **No `window_stack_sitk()`** — CT uses hardcoded Hounsfield Unit windowing (e.g., `window_stack_sitk(img, 50, 100)`).
  T1w MRI has no standardized intensity units; scanner/sequence differences mean the same tissue can
  have wildly different raw values. Windowing is therefore omitted entirely.
- **Otsu thresholding on raw MRI intensities** — Otsu finds the optimal threshold by analyzing the
  histogram shape (the valley between bright brain tissue and dark CSF). It is scanner-agnostic and
  works directly on uncorrected intensities.
- **Adaptive thresholding** for local contrast — the 3rd ventricle is a narrow midline structure;
  global Otsu alone can miss it. A local adaptive threshold (mean-C method, large kernel) is combined
  with the Otsu result for robust ventricle detection.
- **AC/PC coordinates** — OASIS atlas-registered data uses hardcoded T88-space voxel coordinates;
  ABCD data loads AC/PC from per-subject `.txt` files written by Charlie's ART batch script.

## Library imports

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import cv2 as cv
import SimpleITK as sitk
from skimage import measure, filters
import matplotlib.pyplot as plt

from mri_utils import (rotate_image, translate_image, getLargestCC, moving_average,
                       normalize_to_uint8,
                       OASIS_AC_VOXEL, OASIS_PC_VOXEL, parse_acpc_coords, load_acpc_from_txt)

## Configuration

Set `DATASET` to match your data source. The notebook adapts file discovery and AC/PC coordinate
loading accordingly.

Supported datasets:
- **`"oasis1"`** — OASIS-1 cross-sectional (skull-stripped, atlas-registered `.img` files,
  T88 atlas space with hardcoded AC/PC voxel coordinates)
- **`"abcd"`** — ABCD Study T1w scans (ART-corrected `.nii`/`.nii.gz` files with per-subject
  AC/PC `.txt` coordinate files)
- **`"custom"`** — Any NIfTI dataset; supply your own `SCAN_GLOB` and AC/PC loading logic

In [ ]:
# ── Choose your dataset ──────────────────────────────────────────────
DATASET = "oasis1"  # "oasis1", "abcd", or "custom"

# ── Dataset-specific settings ────────────────────────────────────────
if DATASET == "oasis1":
    DATA_DIR = "data/oasis1/disc1"
    DEMOGRAPHICS_PATH = "data/oasis1/oasis_cross-sectional-5708aa0a98d82080.xlsx"
    # OASIS skull-stripped, gain-field corrected, atlas-registered volumes
    SCAN_GLOB = "OAS1_*/PROCESSED/MPRAGE/T88_111/*_masked_gfc.img"
    SUBJECT_ID_FROM_PATH = lambda p: p.split(os.sep)[-5]  # e.g. OAS1_0001_MR1
    # AC/PC .txt files are NOT present for OASIS; hardcoded T88 voxel coords are used instead
    ACPC_TXT_GLOB = None

elif DATASET == "abcd":
    DATA_DIR = "data/abcd"
    DEMOGRAPHICS_PATH = None
    # ART-corrected T1w scans (output from Charlie's ART batch script)
    SCAN_GLOB = "**/*_T1w_acpc.nii*"
    SUBJECT_ID_FROM_PATH = lambda p: os.path.basename(p).split("_ses")[0]  # e.g. sub-NDARINVXXXXXXXX
    # Per-subject AC/PC coordinate files written by ART batch script
    ACPC_TXT_GLOB = "**/*_acpc_coords.txt"

elif DATASET == "custom":
    DATA_DIR = "data/my_dataset"
    DEMOGRAPHICS_PATH = None
    SCAN_GLOB = "**/*.nii.gz"  # adjust to your file naming
    SUBJECT_ID_FROM_PATH = lambda p: os.path.basename(p).replace(".nii.gz", "").replace(".nii", "")
    ACPC_TXT_GLOB = None  # set to your AC/PC file pattern or None to use hardcoded coords

else:
    raise ValueError(f"Unknown DATASET: {DATASET!r}. Use 'oasis1', 'abcd', or 'custom'.")

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Dataset:      {DATASET}")
print(f"Data dir:     {DATA_DIR}")
print(f"Scan pattern: {SCAN_GLOB}")

## Processing parameters

| Parameter | Purpose |
|---|---|
| `cur_thresh` | Curvature threshold for ventricle boundary smoothness check |
| `dec_thresh` | Decreasing-profile threshold — minimum drop required to count as narrowing |
| `ax_gauss_blur_size` | Gaussian kernel size (px) for pre-smoothing the axial slice |
| `ax_adaptive_thresh_size` | Block size for local adaptive threshold — must be odd, large enough to span the 3rd ventricle |
| `ax_adaptive_thresh_mean_thresh` | Constant subtracted from local mean in adaptive threshold (C in OpenCV docs) |
| `v3_roi_col_left_lim_dec` | Pixels to subtract from left ROI boundary (widens ROI leftward) |
| `v3_roi_col_right_lim_inc` | Pixels to add to right ROI boundary (widens ROI rightward) |

In [ ]:
nmax_3vw_processing_params = {
    "cur_thresh": 3,
    "dec_thresh": 0,
    "ax_gauss_blur_size": 3,
    "ax_adaptive_thresh_size": 85,
    "ax_adaptive_thresh_mean_thresh": 7.5,
    "v3_roi_col_left_lim_dec": 7,
    "v3_roi_col_right_lim_inc": 8,
}

## Helper: `get_fm_row()`

Finds the **Foramen of Monro row** — the superior boundary of the 3rd ventricle ROI.

The Foramen of Monro is the junction between the lateral ventricles and the 3rd ventricle.
On the row-sum profile of the ventricle mask, moving superiorly from the AC row (aj),
the 3rd ventricle narrows and then widens again as the lateral ventricles take over.
The foramen row is where the profile stops decreasing — i.e., the first row where
the moving-average derivative is ≥ 0.

In [ ]:
def get_fm_row(ventricles, aj):
    """Finds the Foramen of Monro row by detecting where the vertical profile stops decreasing.

    Parameters
    ----------
    ventricles : np.ndarray
        2D binary ventricle mask for a single axial slice (rows=AP, cols=LR).
    aj : float
        AC y-coordinate (row index) in voxel space — the inferior starting row.

    Returns
    -------
    fm_row : int
        Row index of the Foramen of Monro (superior ROI boundary).
    col_profile : np.ndarray
        Smoothed row-sum profile of the ventricle mask (for diagnostics).
    """
    col_profile = moving_average(np.sum(ventricles, axis=1), 3)
    start_idx = int(round(aj))
    diffs = np.diff(col_profile[start_idx:])
    increasing = np.where(diffs >= 0)[0]
    if len(increasing) == 0:
        fm_row = start_idx
    else:
        fm_row = start_idx + increasing[0]
    return fm_row, col_profile

## Core function: `calc_nmax_3vw()`

Full pipeline for one subject volume. Scans 10 axial slices above the AC–PC plane,
segments the 3rd ventricle on each slice, measures its width, and returns the
maximum normalized width (NMax3VW).

### Algorithm overview

For each candidate slice `k` in `[ak, ak+10)`:

1. **Gaussian blur** — smooth noise before thresholding.
2. **Otsu threshold** — global binarize on raw MRI intensities (no HU windowing).
3. **Adaptive threshold** — local mean-C threshold with a large kernel to catch the
   narrow 3rd ventricle that global Otsu can over- or under-segment.
4. **Background subtraction via floodfill** — floodfill from corner (0,0) removes
   exterior background; the remaining mask is brain-interior only.
5. **V3 ROI** — restrict to columns between `aj` (AC) and `pj` (PC) ± margin parameters.
6. **Foramen of Monro row** — call `get_fm_row()` to find the superior boundary.
7. **Iterative boundary relaxation** — if no ventricle pixels found in the initial ROI,
   progressively widen the column boundaries.
8. **Per-slice width** — median of row-wise pixel counts across ROI rows between
   the Foramen of Monro and PC rows.
9. **Normalize** by total brain width (max column span of brain mask on that slice).
10. **Return** maximum normalized width across all candidate slices.

In [ ]:
def calc_nmax_3vw(axial_data, ak, aj, pj, nmax_3vw_processing_params):
    """Calculates Normalized Maximum 3rd Ventricle Width from a T1w MRI volume.

    Adapts the CT NMax3VW pipeline for MRI. No window_stack_sitk() is used;
    Otsu thresholding runs directly on raw MRI intensities.

    Parameters
    ----------
    axial_data : np.ndarray
        3D array of shape (slices, rows, cols) in axial orientation.
        Slices increase in the superior direction (i.e., higher index = more superior).
    ak : int
        Axial slice index of the AC–PC plane (z-coordinate of AC in voxel space).
    aj : float
        Row index (y-coordinate) of the Anterior Commissure in voxel space.
    pj : float
        Row index (y-coordinate) of the Posterior Commissure in voxel space.
    nmax_3vw_processing_params : dict
        Processing parameters dict (see configuration cell).

    Returns
    -------
    n_3w_width : float or None
        Normalized maximum 3rd ventricle width across candidate slices.
        Returns None if no valid measurement could be obtained.
    """
    gauss_size = nmax_3vw_processing_params["ax_gauss_blur_size"]
    adapt_size = nmax_3vw_processing_params["ax_adaptive_thresh_size"]
    adapt_C    = nmax_3vw_processing_params["ax_adaptive_thresh_mean_thresh"]
    left_dec   = nmax_3vw_processing_params["v3_roi_col_left_lim_dec"]
    right_inc  = nmax_3vw_processing_params["v3_roi_col_right_lim_inc"]

    n_slices = axial_data.shape[0]
    n_rows   = axial_data.shape[1]
    n_cols   = axial_data.shape[2]

    # Columns defining the 3V ROI: between AC (aj) and PC (pj) ± margins.
    # In the axial plane, aj and pj are *row* indices (AP direction).
    # The 3rd ventricle lies between them in the AP direction.
    v3_col_left  = int(min(aj, pj)) - left_dec
    v3_col_right = int(max(aj, pj)) + right_inc
    v3_col_left  = max(0, v3_col_left)
    v3_col_right = min(n_rows - 1, v3_col_right)

    normalized_widths = []

    # Scan slices from ac-plane upward (superior direction)
    for k in range(ak, min(ak + 10, n_slices)):
        ax_slice = axial_data[k, :, :].copy()

        if ax_slice.max() == 0:
            continue

        # ── Step 1: Gaussian blur ────────────────────────────────────────────
        ax_blurred = cv.GaussianBlur(
            ax_slice.astype(np.float32),
            (gauss_size, gauss_size), 0
        )

        # ── Step 2: Otsu threshold on raw MRI intensities ────────────────────
        # Key MRI adaptation: no HU windowing. Otsu finds the valley between
        # dark CSF and bright brain tissue directly from the histogram.
        try:
            otsu_thresh = filters.threshold_otsu(ax_blurred[ax_blurred > 0])
        except ValueError:
            continue
        ax_otsu = (ax_blurred > otsu_thresh).astype(np.uint8)

        # ── Step 3: Adaptive threshold (local mean-C) ────────────────────────
        # Normalise to 0-255 uint8 for OpenCV adaptive threshold
        ax_norm = cv.normalize(ax_blurred, None, 0, 255, cv.NORM_MINMAX).astype(np.uint8)
        ax_adapt = cv.adaptiveThreshold(
            ax_norm,
            255,
            cv.ADAPTIVE_THRESH_MEAN_C,
            cv.THRESH_BINARY,
            adapt_size,
            adapt_C
        )
        ax_adapt = (ax_adapt > 0).astype(np.uint8)

        # Combine: pixel is "brain tissue" if both Otsu AND adaptive agree
        ax_bin = np.logical_and(ax_otsu, ax_adapt).astype(np.uint8)

        # ── Step 4: Background subtraction via floodfill ─────────────────────
        h, w = ax_bin.shape
        flood_mask = np.zeros((h + 2, w + 2), np.uint8)
        ax_flood = ax_bin.copy()
        cv.floodFill(ax_flood, flood_mask, (0, 0), 1)
        # Invert: 0 = background that was flooded, 1 = foreground
        brain_mask = (ax_flood == 0).astype(np.uint8)

        # Ventricle candidate pixels on this slice:
        # dark regions INSIDE the brain (brain_mask=1 but ax_bin=0)
        ventricles_slice = np.logical_and(brain_mask, np.logical_not(ax_bin)).astype(np.uint8)

        # ── Step 5: Restrict to V3 ROI (AP strip between AC and PC rows) ──────
        ventricles_roi = np.zeros_like(ventricles_slice)
        ventricles_roi[v3_col_left:v3_col_right, :] = ventricles_slice[v3_col_left:v3_col_right, :]

        # Further restrict to midline columns (LR center ± small margin)
        # The 3rd ventricle is a midline structure, occupying roughly ±15% of brain width
        brain_cols = np.where(brain_mask.sum(axis=0) > 0)[0]
        if len(brain_cols) < 2:
            continue
        brain_width = brain_cols[-1] - brain_cols[0] + 1
        midline_col = (brain_cols[0] + brain_cols[-1]) // 2
        v3_col_mid_margin = max(10, int(brain_width * 0.12))
        v3_col_mid_left  = max(0, midline_col - v3_col_mid_margin)
        v3_col_mid_right = min(n_cols - 1, midline_col + v3_col_mid_margin)

        ventricles_v3 = np.zeros_like(ventricles_roi)
        ventricles_v3[v3_col_left:v3_col_right, v3_col_mid_left:v3_col_mid_right] = (
            ventricles_roi[v3_col_left:v3_col_right, v3_col_mid_left:v3_col_mid_right]
        )

        # ── Step 6: Foramen of Monro row ──────────────────────────────────────
        fm_row, _ = get_fm_row(ventricles_v3, aj)
        pc_row = int(round(pj))

        # ROI rows: from pc_row (inferior) to fm_row (superior)
        row_top    = min(fm_row, pc_row)
        row_bottom = max(fm_row, pc_row)

        if row_top >= row_bottom:
            # Degenerate ROI — use a small window around aj
            row_top    = max(0, int(aj) - 5)
            row_bottom = min(n_rows - 1, int(aj) + 5)

        # ── Step 7: Iterative boundary relaxation if V3 not found ─────────────
        roi_pixels = ventricles_v3[row_top:row_bottom, :]
        relax_attempts = 0
        while roi_pixels.sum() == 0 and relax_attempts < 5:
            v3_col_mid_margin = int(v3_col_mid_margin * 1.3)
            v3_col_mid_left  = max(0, midline_col - v3_col_mid_margin)
            v3_col_mid_right = min(n_cols - 1, midline_col + v3_col_mid_margin)
            ventricles_v3 = np.zeros_like(ventricles_roi)
            ventricles_v3[v3_col_left:v3_col_right, v3_col_mid_left:v3_col_mid_right] = (
                ventricles_roi[v3_col_left:v3_col_right, v3_col_mid_left:v3_col_mid_right]
            )
            roi_pixels = ventricles_v3[row_top:row_bottom, :]
            relax_attempts += 1

        if roi_pixels.sum() == 0:
            # No 3rd ventricle pixels found on this slice — skip
            continue

        # ── Step 8: Per-slice 3V width = median of row-wise pixel counts ──────
        row_widths = roi_pixels.sum(axis=1)  # sum across cols for each row
        nonzero_rows = row_widths[row_widths > 0]
        if len(nonzero_rows) == 0:
            continue
        v3_width_px = float(np.median(nonzero_rows))

        # ── Step 9: Normalize by brain width ─────────────────────────────────
        norm_v3_width = v3_width_px / brain_width
        normalized_widths.append(norm_v3_width)

    # ── Step 10: Return maximum across all candidate slices ───────────────────
    if len(normalized_widths) == 0:
        return None
    n_3w_width = float(np.max(normalized_widths))
    return n_3w_width

## AC/PC coordinate loading

`get_acpc_for_subject()` returns both the AC and PC voxel indices needed by `calc_nmax_3vw()`.

- **OASIS**: All disc-1 volumes are registered to the T88 atlas. AC and PC sit at fixed voxel
  positions (`OASIS_AC_VOXEL`, `OASIS_PC_VOXEL`) in that space. No per-subject file needed.
- **ABCD**: Each subject has an `*_acpc_coords.txt` file written by Charlie's ART batch script.
  The file contains two lines — AC coordinates and PC coordinates in world space.
  We convert world → voxel using SimpleITK's `TransformPhysicalPointToIndex`.

Returns `(ak, ai, aj, pk, pi, pj)` where:
- `ak, ai, aj` = AC voxel indices (z, x, y)
- `pk, pi, pj` = PC voxel indices (z, x, y)

In [ ]:
def get_acpc_for_subject(subject_id, scan_path, dataset):
    """Returns AC and PC voxel coordinates for a subject.

    Parameters
    ----------
    subject_id : str
        Subject identifier string.
    scan_path : str
        Absolute path to the subject's MRI volume.
    dataset : str
        One of 'oasis1', 'abcd', 'custom'.

    Returns
    -------
    ak, ai, aj : int
        AC voxel indices (z-slice, x-col, y-row).
    pk, pi, pj : int
        PC voxel indices (z-slice, x-col, y-row).

    Raises
    ------
    FileNotFoundError
        If ABCD AC/PC coordinate file cannot be located.
    ValueError
        If dataset is unsupported and no fallback is available.
    """
    if dataset == "oasis1":
        # T88 atlas-space hardcoded voxel coordinates (x, y, z) = (i, j, k)
        # OASIS_AC_VOXEL = (85, 110, 73) → (x=85, y=110, z=73)
        ai, aj, ak = OASIS_AC_VOXEL  # (x, y, z)
        pi, pj, pk = OASIS_PC_VOXEL  # (x, y, z)
        return ak, ai, aj, pk, pi, pj

    elif dataset == "abcd":
        # Look for *_acpc_coords.txt in the same directory as the scan
        scan_dir = os.path.dirname(scan_path)
        txt_files = glob.glob(os.path.join(scan_dir, "*_acpc_coords.txt"))
        if not txt_files:
            # Try subject-level directory one level up
            txt_files = glob.glob(os.path.join(os.path.dirname(scan_dir), "*_acpc_coords.txt"))
        if not txt_files:
            raise FileNotFoundError(
                f"No *_acpc_coords.txt found for {subject_id} near {scan_path}"
            )

        ac_world, pc_world = load_acpc_from_txt(txt_files[0])

        # Convert world coordinates → voxel indices using the scan's geometry
        img = sitk.ReadImage(scan_path)
        ac_idx = img.TransformPhysicalPointToIndex(ac_world)  # returns (x, y, z)
        pc_idx = img.TransformPhysicalPointToIndex(pc_world)

        ai, aj, ak = int(ac_idx[0]), int(ac_idx[1]), int(ac_idx[2])
        pi, pj, pk = int(pc_idx[0]), int(pc_idx[1]), int(pc_idx[2])
        return ak, ai, aj, pk, pi, pj

    else:
        # Custom dataset: fall back to midpoint heuristic if no coords available
        # Users should override this block with dataset-specific logic
        img = sitk.ReadImage(scan_path)
        sz = img.GetSize()  # (width, height, depth) = (x, y, z)
        ak = sz[2] // 2
        ai, aj = sz[0] // 2, sz[1] // 2
        pk = ak
        pi, pj = ai, int(aj * 0.8)
        print(f"  [WARNING] Using heuristic AC/PC for {subject_id} — override get_acpc_for_subject() for accuracy.")
        return ak, ai, aj, pk, pi, pj

## Discover scans

Uses the `SCAN_GLOB` pattern from the configuration cell to enumerate all volumes in `DATA_DIR`.

In [ ]:
scan_files = sorted(glob.glob(os.path.join(DATA_DIR, SCAN_GLOB), recursive=True))

scan_df = pd.DataFrame({
    "subject_id": [SUBJECT_ID_FROM_PATH(p) for p in scan_files],
    "scan_path":  scan_files,
})

print(f"Found {len(scan_df)} scans in {DATA_DIR}")
scan_df.head()

## Single-subject test

Run the full pipeline on the first subject to verify orientation, AC/PC loading, and the
3rd ventricle segmentation before committing to a batch run.

**What to expect:**
- NMax3VW for a healthy young adult ≈ 0.01–0.04 (3rd ventricle is a narrow slit).
- Older subjects or those with enlarged ventricles may reach 0.08–0.15.
- If `None` is returned, the 3rd ventricle was not detected — inspect the AC/PC coordinates
  and processing parameters.

In [ ]:
# ── Load first subject ────────────────────────────────────────────────
test_path = scan_df.iloc[0]["scan_path"]
test_id   = scan_df.iloc[0]["subject_id"]
print(f"Testing on: {test_id}")
print(f"Path:       {test_path}")

# ── Load and orient volume ────────────────────────────────────────────
img = sitk.ReadImage(test_path)
print(f"\nOriginal size:    {img.GetSize()}")
print(f"Original spacing: {img.GetSpacing()}")

img_rotated = rotate_image(img)
ax_data = sitk.GetArrayFromImage(img_rotated)  # shape: (slices, rows, cols)
print(f"Array shape:      {ax_data.shape}")
print(f"Intensity range:  [{ax_data.min()}, {ax_data.max()}]")

# ── Load AC/PC coordinates ────────────────────────────────────────────
ak, ai, aj, pk, pi, pj = get_acpc_for_subject(test_id, test_path, DATASET)
print(f"\nAC voxel (z={ak}, x={ai}, y={aj})")
print(f"PC voxel (z={pk}, x={pi}, y={pj})")

# ── Visualize AC-PC slice for sanity check ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].set_title(f"AC-PC slice (k={ak})")
axes[0].imshow(ax_data[ak, :, :], cmap="gray")
axes[0].axhline(aj, color="cyan",  linewidth=1, label=f"AC row (aj={aj})")
axes[0].axhline(pj, color="yellow", linewidth=1, label=f"PC row (pj={pj})")
axes[0].legend(fontsize=8)

axes[1].set_title(f"Slice ak+5 ({ak+5})")
axes[1].imshow(ax_data[min(ak + 5, ax_data.shape[0] - 1), :, :], cmap="gray")

axes[2].set_title("Intensity histogram (non-zero)")
axes[2].hist(ax_data[ax_data > 0].flatten(), bins=100)
axes[2].set_xlabel("Raw MRI intensity")
axes[2].set_ylabel("Voxel count")

plt.tight_layout()
plt.show()

# ── Run NMax3VW ───────────────────────────────────────────────────────
nmax3vw = calc_nmax_3vw(ax_data, ak, aj, pj, nmax_3vw_processing_params)
print(f"\nNMax3VW for {test_id}: {nmax3vw}")

## Batch processing

Run NMax3VW on all discovered subjects. Visualization is suppressed (`visualize=False` equivalent
— no per-slice plots). Errors are caught and recorded as `NaN` so the loop always completes.

In [ ]:
results = []

for idx, row in scan_df.iterrows():
    subject_id = row["subject_id"]
    vol_path   = row["scan_path"]

    try:
        # Load and orient
        img     = sitk.ReadImage(vol_path)
        ax_data = sitk.GetArrayFromImage(rotate_image(img))

        # AC/PC coordinates
        ak, ai, aj, pk, pi, pj = get_acpc_for_subject(subject_id, vol_path, DATASET)

        # Compute NMax3VW
        nmax3vw = calc_nmax_3vw(ax_data, ak, aj, pj, nmax_3vw_processing_params)

        results.append({
            "subject_id": subject_id,
            "nmax3vw":    nmax3vw,
            "ak":         ak,
            "aj":         aj,
            "pj":         pj,
        })
        print(f"[{idx+1:>3}/{len(scan_df)}] {subject_id}: NMax3VW={nmax3vw}")

    except Exception as exc:
        print(f"[{idx+1:>3}/{len(scan_df)}] {subject_id}: ERROR — {exc}")
        results.append({
            "subject_id": subject_id,
            "nmax3vw":    np.nan,
            "ak":         np.nan,
            "aj":         np.nan,
            "pj":         np.nan,
        })

results_df = pd.DataFrame(results)
print(f"\nProcessed {len(results_df)} subjects")
print(f"Valid measurements: {results_df['nmax3vw'].notna().sum()}")
results_df.head()

## Results

Save results to CSV and display summary statistics and distribution plots.

**Expected distribution:**
- Healthy young adults (OASIS CDR=0): NMax3VW ≈ 0.01–0.04
- Older healthy adults: NMax3VW ≈ 0.02–0.06
- Mild dementia (CDR 0.5–1.0): NMax3VW often elevated toward 0.06–0.12
- Right-skewed distribution with a long tail toward higher values is typical

In [ ]:
# ── Save to CSV ───────────────────────────────────────────────────────
output_csv = os.path.join(OUTPUT_DIR, f"nmax3vw_{DATASET}.csv")
results_df.to_csv(output_csv, index=False)
print(f"Results saved to {output_csv}")

# ── Optionally merge demographics ─────────────────────────────────────
if DEMOGRAPHICS_PATH and os.path.exists(DEMOGRAPHICS_PATH):
    ext = os.path.splitext(DEMOGRAPHICS_PATH)[1].lower()
    if ext in (".xlsx", ".xls"):
        demo_df = pd.read_excel(DEMOGRAPHICS_PATH)
    elif ext == ".csv":
        demo_df = pd.read_csv(DEMOGRAPHICS_PATH)
    elif ext in (".tsv", ".txt"):
        demo_df = pd.read_csv(DEMOGRAPHICS_PATH, sep="\t")
    else:
        demo_df = None

    if demo_df is not None:
        id_col = next(
            (c for c in demo_df.columns if c.upper() in ("ID", "SUBJECT_ID", "SUBJECTKEY", "SRC_SUBJECT_ID")),
            None
        )
        if id_col:
            keep_cols = [id_col] + [
                c for c in demo_df.columns
                if c != id_col and c in ("Age", "M/F", "CDR", "Sex", "age", "sex", "interview_age")
            ]
            results_df = results_df.merge(
                demo_df[keep_cols],
                left_on="subject_id", right_on=id_col,
                how="left"
            )
            print(f"Merged demographics ({id_col} → subject_id)")
else:
    demo_df = None

# ── Summary statistics ────────────────────────────────────────────────
valid = results_df["nmax3vw"].dropna()
print(f"\nNMax3VW summary (n={len(valid)}):")
print(valid.describe().round(5))

# ── Plots ─────────────────────────────────────────────────────────────
has_cdr = "CDR" in results_df.columns and results_df["CDR"].notna().any()
n_plots = 2 if not has_cdr else 3
fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 5))

# Plot 1: Distribution histogram
axes[0].hist(valid, bins=25, edgecolor="black", color="steelblue")
axes[0].axvline(valid.mean(),   color="red",    linestyle="--", label=f"Mean={valid.mean():.4f}")
axes[0].axvline(valid.median(), color="orange", linestyle="--", label=f"Median={valid.median():.4f}")
axes[0].set_xlabel("NMax3VW")
axes[0].set_ylabel("Count")
axes[0].set_title("NMax3VW Distribution")
axes[0].legend()

# Plot 2: Box plot (overall or by CDR)
if has_cdr:
    cdr_groups = sorted(results_df["CDR"].dropna().unique())
    box_data   = [results_df.loc[results_df["CDR"] == c, "nmax3vw"].dropna().values for c in cdr_groups]
    axes[1].boxplot(box_data, labels=[f"CDR {c}" for c in cdr_groups])
    axes[1].set_ylabel("NMax3VW")
    axes[1].set_title("NMax3VW by Dementia Severity")
else:
    axes[1].boxplot(valid)
    axes[1].set_ylabel("NMax3VW")
    axes[1].set_title("NMax3VW Box Plot")
    axes[1].set_xticks([])

# Plot 3: Histogram by CDR group (if available)
if has_cdr:
    for cdr_val in sorted(results_df["CDR"].dropna().unique()):
        subset = results_df.loc[results_df["CDR"] == cdr_val, "nmax3vw"].dropna()
        if len(subset) > 0:
            axes[2].hist(subset, bins=15, alpha=0.55, label=f"CDR {cdr_val} (n={len(subset)})")
    axes[2].set_xlabel("NMax3VW")
    axes[2].set_ylabel("Count")
    axes[2].set_title("NMax3VW by CDR Group")
    axes[2].legend()

plt.tight_layout()
plt.show()

if has_cdr:
    print("\nNMax3VW by CDR group:")
    print(results_df.groupby("CDR")["nmax3vw"].describe().round(5))